# 02 KR2036 網路分析（kwak_full 情境）

改寫自上游 `notebooks/network_analysis.ipynb`，保留原本的章節順序，全部改跑
**kwak_full**（10 節點 / 3H）情境。

**與原版的主要差異**

| 項目 | 上游原版 | 韓國版 |
|---|---|---|
| 網路檔定位 | wildcard 搜尋 `elec_s{simpl}_{clusters}_ec_l{ll}_{opts}.nc` | 直接讀 `_kr_common.SCENARIOS` 的絕對路徑，不猜檔名 |
| 地圖投影 | `ccrs.EqualEarth()` + 自動 bounds | `ccrs.PlateCarree()` + 固定 `KR_EXTENT`（否則離岸風節點會把畫面拉到海上）|
| load shedding | 只在部分圖排除 | `load_network()` 一律先移除，所有容量／發電／圓餅都不含虛擬機組 |
| 風光潛力圖 | 只有 onwind / solar | 另加 **offwind-ac / offwind-dc**（用離岸 voronoi 區域）|
| 國家過濾 | `filter(regex="NG *")` 字串比對 | 不過濾（全網只有 KR）|

## 0. 環境設定與載入網路

In [ ]:
import sys
import warnings
from pathlib import Path

_here = Path.cwd()
for cand in (_here, _here / "notebooks_kr", _here.parent):
    if (cand / "_kr_common.py").exists():
        sys.path.insert(0, str(cand))
        break

import _kr_common as K

K.setup_matplotlib()

import cartopy
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

SCEN = "kwak_full"
info = K.SCENARIOS[SCEN]
print("情境：", info["long_label"])
print("網路檔：", info["path"])

### 0.1 load shedding 虛擬機組

`config.yaml` 開了 `load_shedding`，網路裡有 10 台虛擬機組（每個節點一台）。
容量高達 137 GW，如果不排除，所有容量圖與圓餅都會被它灌爆。
下面先量出它的規模再移除，之後所有分析都不含這些機組。

In [ ]:
n_raw = K.load_network(SCEN, drop_load_shedding=False)
shed = n_raw.generators[n_raw.generators.carrier.str.contains("load", case=False)]
shed_mwh = (
    n_raw.generators_t.p[shed.index]
    .mul(n_raw.snapshot_weightings.generators, axis=0)
    .sum()
    .sum()
)
print(f"load shedding 機組：{len(shed)} 台，p_nom 合計 {shed.p_nom.sum() / 1e3:,.1f} GW")
print(f"全年實際動用：{shed_mwh:,.2f} MWh（= {shed_mwh / 1e6:.9f} TWh）")
print("→ 容量很大但幾乎沒用到，代表系統充裕度足夠；以下分析一律排除。")

n = K.load_network(SCEN)  # 預設已排除 load shedding

## 1. 資料載入檢查

In [ ]:
comp_rows = []
for comp in n.iterate_components(list(n.components.keys())[2:]):
    if len(comp.df):
        comp_rows.append({"元件": comp.name, "筆數": len(comp.df)})
components = pd.DataFrame(comp_rows).set_index("元件")

print(f"時間解析度：{len(n.snapshots)} 個 snapshot（權重 {n.snapshot_weightings.generators.unique()} 小時）")
print(f"起訖：{n.snapshots[0]} → {n.snapshots[-1]}")
print(f"DC link 數：{len(n.links)}（10 節點下濟州併入本土叢集，因此為空）")
components

## 2. 區域圖（voronoi 叢集區域）

10 節點的陸域 voronoi 區域，以及離岸風可用的離岸區域。
**濟州在 10 節點下併入本土叢集**，要到 30 節點才會切成獨立節點。

In [ ]:
regions_on = K.load_regions(10, "onshore")
regions_off = K.load_regions(10, "offshore")

fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent(K.KR_EXTENT, crs=ccrs.PlateCarree())

regions_off.plot(
    ax=ax, facecolor="#cfe3f2", edgecolor="#7fa8c9", linewidth=0.5,
    alpha=0.6, transform=ccrs.PlateCarree(), zorder=1,
)
regions_on.plot(
    ax=ax, facecolor="none", edgecolor="#333333", linewidth=1.0,
    transform=ccrs.PlateCarree(), zorder=2,
)
regions_on.plot(
    ax=ax, column=regions_on.index.to_series(), cmap="Pastel2",
    alpha=0.75, transform=ccrs.PlateCarree(), zorder=1,
)

for name, row in regions_on.iterrows():
    ax.annotate(
        name, xy=(row.x, row.y), ha="center", va="center", fontsize=9,
        fontweight="bold", transform=ccrs.PlateCarree(), zorder=4,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
    )

ax.set_title(f"10 節點叢集區域（陸域 {len(regions_on)} 區 / 離岸 {len(regions_off)} 區）", fontsize=13, pad=10)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
gl.top_labels = False
gl.right_labels = False

K.savefig("02_fig1_regions_10n")
plt.show()

area = regions_on.to_crs(K.KR_EQUAL_AREA_CRS).geometry.area * 1e-6
print(f"陸域區域總面積：{area.sum():,.0f} km²（南韓國土約 100,200 km²）")

## 3. 容量地圖

### 3.1 繪圖設定

`bus_size_factor` 與 `linewidth_factor` 沿用 `scripts/plot_network.py` 已修好的韓國尺度
（2e6 / 1e4），配色取 `config.yaml` 的 `plotting.tech_colors`，與
`results/compare/kwak_full_vs_paper/` 既有圖表同源。

In [ ]:
BUS_SIZE_FACTOR = 2e6
LINEWIDTH_FACTOR = 1e4


def capacity_map(n, attr, title, fname):
    gen = n.generators.groupby(["bus", "carrier"])[attr].sum()
    sto_attr = attr if attr in n.storage_units.columns else "p_nom"
    sto = n.storage_units.groupby(["bus", "carrier"])[sto_attr].sum()
    sizes = pd.concat([gen, sto])
    sizes = sizes[sizes > 0]

    carriers = sizes.index.get_level_values(1).unique()
    colors = {c: K.carrier_color(c) for c in carriers}

    line_attr = "s_nom_opt" if attr == "p_nom_opt" else "s_nom"

    fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={"projection": ccrs.PlateCarree()})
    n.plot(
        ax=ax,
        bus_sizes=sizes / BUS_SIZE_FACTOR,
        bus_colors=colors,
        bus_alpha=0.85,
        line_widths=n.lines[line_attr] / LINEWIDTH_FACTOR,
        line_colors="#70af1d",
        link_widths=0,  # 10 節點沒有 DC link，n.links 為空
        geomap=True,
        color_geomap={"ocean": "white", "land": "whitesmoke"},
        boundaries=K.KR_EXTENT,
    )

    add_legend_circles(
        ax,
        [s / BUS_SIZE_FACTOR for s in (5e3, 20e3)],
        ["5 GW", "20 GW"],
        legend_kw=dict(loc="upper left", bbox_to_anchor=(0.0, 1.0), frameon=True, labelspacing=1.8),
    )
    add_legend_lines(
        ax,
        [s / LINEWIDTH_FACTOR for s in (5e3, 20e3)],
        ["5 GW", "20 GW"],
        legend_kw=dict(loc="upper left", bbox_to_anchor=(0.0, 0.78), frameon=True),
    )
    order = sizes.groupby(level=1).sum().sort_values(ascending=False).index
    add_legend_patches(
        ax,
        [K.carrier_color(x) for x in order],
        [f"{K.carrier_label(x)}  {sizes.groupby(level=1).sum()[x] / 1e3:,.1f} GW" for x in order],
        legend_kw=dict(loc="lower left", bbox_to_anchor=(1.0, 0.0), frameon=False, fontsize=9),
    )

    ax.set_title(title, fontsize=13, pad=10)
    K.savefig(fname)
    plt.show()
    return sizes

### 3.2 現有容量（`p_nom`，最佳化前）

In [ ]:
sizes_installed = capacity_map(
    n, "p_nom",
    f"KR2036 {info['label']}：現有裝置容量（p_nom）",
    "02_fig2_capacity_map_installed",
)
print(f"現有容量合計：{sizes_installed.sum() / 1e3:,.1f} GW")

### 3.3 最佳化後容量（`p_nom_opt`）

In [ ]:
sizes_optimal = capacity_map(
    n, "p_nom_opt",
    f"KR2036 {info['label']}：最佳化後裝置容量（p_nom_opt）",
    "02_fig3_capacity_map_optimal",
)
print(f"最佳化後容量合計：{sizes_optimal.sum() / 1e3:,.1f} GW")

## 4. 容量圓餅：最佳化前後對照

左圖是既有機組（`p_nom`），右圖是最佳化後（`p_nom_opt`）。

In [ ]:
def capacity_by_carrier(n, attr):
    sto_attr = attr if attr in n.storage_units.columns else "p_nom"
    s = pd.concat([
        n.generators.groupby("carrier")[attr].sum(),
        n.storage_units.groupby("carrier")[sto_attr].sum(),
    ]).div(1e3)
    return s[s > 0].sort_values(ascending=False)


cap_installed = capacity_by_carrier(n, "p_nom")
cap_optimal = capacity_by_carrier(n, "p_nom_opt")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, data, sub in zip(axes, [cap_installed, cap_optimal], ["現有（p_nom）", "最佳化後（p_nom_opt）"]):
    colors = [K.carrier_color(x) for x in data.index]
    wedges, _ = ax.pie(data, colors=colors, startangle=90, counterclock=False)
    ax.set_title(f"{sub}　合計 {data.sum():,.1f} GW", fontsize=12)
    ax.legend(
        wedges,
        [f"{K.carrier_label(i)}  {v:,.1f} GW（{100 * v / data.sum():.1f}%）" for i, v in data.items()],
        loc="upper center", bbox_to_anchor=(0.5, -0.02), fontsize=9, frameon=False,
    )
    ax.set_aspect("equal")
    ax.grid(False)

fig.suptitle(f"KR2036 {info['label']}：裝置容量結構（已排除 load shedding）", fontsize=14)
K.savefig("02_fig4_capacity_pie")
plt.show()

pd.DataFrame({"現有_GW": cap_installed, "最佳化後_GW": cap_optimal}).fillna(0).round(2)

## 5. 容量擴充量長條圖

`n.statistics.optimal_capacity()` − `n.statistics.installed_capacity()`。

注意 `n.statistics` 回傳的 carrier 是 **nice_name**（如 `Combined-Cycle Gas`），
`_kr_common.carrier_key()` 會反查回原始 carrier 以對上配色與中文標籤。
離岸風（DC）沒有既有機組，相減時要 `fill_value=0`，否則會變成 NaN。

In [ ]:
optimal = n.statistics.optimal_capacity(comps=["Generator"]).droplevel(0).div(1e3)
installed = n.statistics.installed_capacity(comps=["Generator"]).droplevel(0).div(1e3)
expansion = optimal.sub(installed, fill_value=0).sort_values(ascending=False)
expansion = expansion[expansion.abs() > 1e-6]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(
    [K.carrier_label(i) for i in expansion.index],
    expansion.values,
    color=[K.carrier_color(i) for i in expansion.index],
    edgecolor="white", linewidth=0.8,
)
ax.axhline(0, color="#444", linewidth=0.8)
ax.set_title(f"KR2036 {info['label']}：發電容量擴充量（最佳化後 − 既有）", fontsize=13, pad=10)
ax.set_ylabel("容量擴充 [GW]")
ax.tick_params(axis="x", rotation=30)
for i, v in enumerate(expansion.values):
    ax.text(i, v + (0.6 if v >= 0 else -1.2), f"{v:,.1f}", ha="center", fontsize=9)

K.savefig("02_fig5_capacity_expansion")
plt.show()

pd.DataFrame({
    "既有_GW": installed.reindex(expansion.index).fillna(0),
    "最佳化後_GW": optimal.reindex(expansion.index).fillna(0),
    "擴充_GW": expansion,
}).round(2)

## 6. 能量平衡（`n.statistics.energy_balance()`）

本機 pypsa 0.30.3 的 `energy_balance()` 回傳三層 index
（`component` / `carrier` / `bus_carrier`），carrier 一樣是 nice_name。
正值為供給、負值為需求（`-` 是負載，儲能充電也是負的）。

In [ ]:
eb = n.statistics.energy_balance()
eb_carrier = eb.groupby("carrier").sum().div(1e6)  # MWh → TWh
eb_carrier = eb_carrier[eb_carrier.abs() > 1e-9].sort_values(ascending=False)

supply = eb_carrier[eb_carrier > 0]
demand = eb_carrier[eb_carrier < 0]

fig, ax = plt.subplots(figsize=(11, 4.2))
left = 0.0
for name, val in supply.items():
    ax.barh(0, val, left=left, color=K.carrier_color(name), edgecolor="white", linewidth=0.6, height=0.5)
    left += val
left = 0.0
for name, val in demand.items():
    ax.barh(-0.75, val, left=left, color=K.carrier_color(name), edgecolor="white", linewidth=0.6, height=0.5)
    left += val

ax.axvline(0, color="#444", linewidth=0.8)
ax.set_yticks([0, -0.75])
ax.set_yticklabels(["供給", "需求"])
ax.set_xlabel("TWh")
ax.set_title(f"KR2036 {info['label']}：年度能量平衡", fontsize=13, pad=10)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{int(x):,}"))
ax.grid(axis="x", linestyle=":", alpha=0.6)

handles = [
    Patch(
        facecolor=K.carrier_color(i),
        label=f"{'負載' if i == '-' else K.carrier_label(i)}  {v:,.1f} TWh",
    )
    for i, v in eb_carrier.items()
]
ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, frameon=False)

K.savefig("02_fig6_energy_balance")
plt.show()

### 6.1 能量平衡表（TWh）

In [ ]:
eb_table = pd.DataFrame({"TWh": eb_carrier})
eb_table.index = [K.carrier_label(i) if i != "-" else "負載" for i in eb_table.index]
gen_total = supply.sum()
eb_table["佔總發電%"] = (eb_table["TWh"] / gen_total * 100).where(eb_table["TWh"] > 0)

K.FIG_DIR.mkdir(parents=True, exist_ok=True)
eb_table.round(3).to_csv(K.FIG_DIR / "02_table_energy_balance_TWh.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_energy_balance_TWh.csv")
print(f"總發電 {gen_total:,.1f} TWh　總負載 {-demand.get('-', 0):,.1f} TWh")
eb_table.round(2)

## 7. 儲能現況：抽蓄無法儲能，而它的充電行為是「必發過剩」的獨立證據

這一節有兩個目的：一是避免誤讀前面幾張圖（電池與抽蓄小到看不見，是**真的幾乎為零**，
不是繪圖漏畫）；二是把抽蓄的異常行為講清楚——它**不是單純的無作用**，
而是一個可以用來佐證「熱機組必發造成過剩」的觀測值。

### 7.1 各儲能載體的實際運轉

In [ ]:
su = n.storage_units
w = n.snapshot_weightings.stores
rows = []
for carrier in ["battery", "PHS", "hydro"]:
    idx = su.index[su.carrier == carrier]
    if not len(idx):
        continue
    pc = n.storage_units_t.p[idx]
    rows.append({
        "載體": K.carrier_label(carrier),
        "p_nom_opt_MW": su.loc[idx, "p_nom_opt"].sum(),
        "max_hours": su.loc[idx, "max_hours"].unique()[0],
        "放電_GWh": pc.clip(lower=0).mul(w, axis=0).sum().sum() / 1e3,
        "充電_GWh": -pc.clip(upper=0).mul(w, axis=0).sum().sum() / 1e3,
    })
storage = pd.DataFrame(rows).set_index("載體")
storage["淨_GWh"] = storage["放電_GWh"] - storage["充電_GWh"]
storage.round(2)

### 7.2 抽蓄到底在做什麼：同時充放電，而不是 spill

`max_hours = 0` 讓儲能容量 `p_nom × max_hours` 歸零，SOC 全年恆為 0。
但模型並沒有因此讓抽蓄停擺，而是讓它**在同一時段同時充電與放電**，
靠來回效率把電就地損耗掉。下面驗證這個機制：`spill` 實測為 0，
真正成立的是 `η_store × p_store = p_dispatch ÷ η_dispatch`。

In [ ]:
def phs_profile(key):
    m = K.load_network(key)
    su_ = m.storage_units
    w_ = m.snapshot_weightings.stores
    idx = su_.index[su_.carrier == "PHS"]
    t = m.storage_units_t

    gross_store = float(t.p_store[idx].mul(w_, axis=0).sum().sum() / 1e3)
    gross_dispatch = float(t.p_dispatch[idx].mul(w_, axis=0).sum().sum() / 1e3)
    spill = (
        float(t.spill.reindex(columns=idx).fillna(0).mul(w_, axis=0).sum().sum() / 1e3)
        if len(t.spill.columns) else 0.0
    )
    eff_s = float(su_.loc[idx, "efficiency_store"].iloc[0])
    eff_d = float(su_.loc[idx, "efficiency_dispatch"].iloc[0])

    # 充電量加權的節點邊際價格（未加權會被每個時段都存在的微量殘留稀釋）
    price = m.buses_t.marginal_price
    parts = []
    for u in idx:
        ch = (-t.p[u]).clip(lower=0) * w_
        msk = ch > 1e-9
        parts.append(pd.DataFrame({"gwh": ch[msk] / 1e3, "price": price.loc[msk, su_.at[u, "bus"]]}))
    d = pd.concat(parts)
    tot = d.gwh.sum()

    return {
        "p_nom_opt_MW": float(su_.loc[idx, "p_nom_opt"].sum()),
        "max_hours": float(su_.loc[idx, "max_hours"].iloc[0]),
        "SOC 絕對值最大_MWh": float(t.state_of_charge[idx].abs().max().max()),
        "毛充電 p_store_GWh": gross_store,
        "毛放電 p_dispatch_GWh": gross_dispatch,
        "spill_GWh": spill,
        "淨吸收（來回損失）_GWh": gross_store - gross_dispatch,
        "能量平衡檢核 η_s×store": eff_s * gross_store,
        "能量平衡檢核 dispatch÷η_d": gross_dispatch / eff_d,
        "充電量加權電價 中位數": float(np.interp(
            0.5, np.cumsum(d.sort_values("price").gwh) / tot, d.sort_values("price").price)),
        "充電量落在電價<1 的比例%": float(100 * d.gwh[d.price < 1].sum() / tot),
    }


phs = pd.DataFrame({k: phs_profile(k) for k in ["baseline", "kwak_full"]})
_energy_rows = ["毛充電 p_store_GWh", "毛放電 p_dispatch_GWh", "淨吸收（來回損失）_GWh"]
phs["倍數 (kwak/base)"] = np.nan
phs.loc[_energy_rows, "倍數 (kwak/base)"] = (
    phs.loc[_energy_rows, "kwak_full"] / phs.loc[_energy_rows, "baseline"]
)
phs.round(3)

### 7.3 必發過剩的對照實驗

如果 7.2 的來回損失真的來自「低需求時段電沒有去處」，它應該集中在**邊際價格崩到接近 0**
的時段，而且應該與 VRE 棄電同時發生。

但**直接拿基準情境對 kwak_full 是不受控的比較**：兩者的年需求（667 對 707 TWh）與
VRE 裝置量（102.9 對 132.8 GW）都不同，這兩項都會往「增加過剩」的同方向推。
要分離必發限制的單獨效果，必須找需求與 VRE 都相同、只差必發設定的情境。

repo 裡剛好有：`10n_3H_uc_lyden` 與 `10n_3H_ramp` 的需求（667.3 TWh）與
VRE 容量（約 102.9 GW）和基準情境一致，差別只在熱機組必發下限。
其中 `uc_lyden` 用的正是 kwak_full 的 `p_min_pu`（燃煤 0.357 / 核能 0.45）。

In [ ]:
CONTROL = {
    "① 對照組（無必發）": K.SCENARIOS["baseline"]["path"],
    "② +必發 煤.357/核.45": K.ROOT / "solved/10n_3H_uc_lyden.nc",
    "③ +必發 煤.35/核.70": K.ROOT / "solved/10n_3H_ramp.nc",
    "④ kwak_full（必發+高VRE+高需求）": K.SCENARIOS["kwak_full"]["path"],
}


def surplus_profile(path):
    m = pypsa.Network(str(path))
    shed = m.generators.index[m.generators.carrier.str.contains("load", case=False)]
    m.mremove("Generator", shed)

    w_ = m.snapshot_weightings.stores
    g = m.generators
    vre = g.index[g.carrier.isin(["solar", "onwind", "offwind-ac", "offwind-dc"])]
    avail = (
        m.generators_t.p_max_pu.reindex(columns=vre).fillna(g.loc[vre, "p_max_pu"])
        * g.loc[vre, "p_nom_opt"]
    )
    curt = (avail - m.generators_t.p[vre]).clip(lower=0)

    su_ = m.storage_units
    phs_i = su_.index[su_.carrier == "PHS"]
    t = m.storage_units_t
    net_absorb = float(
        (t.p_store[phs_i].mul(w_, axis=0).sum().sum() - t.p_dispatch[phs_i].mul(w_, axis=0).sum().sum()) / 1e3
    )
    charge_mw = (-t.p[phs_i].sum(axis=1)).clip(lower=0)
    price = m.buses_t.marginal_price.mean(axis=1)

    avail_gwh = float(avail.mul(w_, axis=0).sum().sum() / 1e3)
    curt_gwh = float(curt.mul(w_, axis=0).sum().sum() / 1e3)

    return {
        "年需求_TWh": float(m.loads_t.p_set.sum(axis=1).mul(m.snapshot_weightings.generators).sum() / 1e6),
        "VRE 容量_GW": float(g.loc[vre, "p_nom_opt"].sum() / 1e3),
        "燃煤 p_min_pu": float(g[g.carrier == "coal"].p_min_pu.mean()),
        "核能 p_min_pu": float(g[g.carrier == "nuclear"].p_min_pu.mean()),
        "PHS 淨吸收_GWh": net_absorb,
        "VRE 棄電_GWh": curt_gwh,
        "VRE 棄電率%": 100 * curt_gwh / avail_gwh,
        "電價 p10": float(price.quantile(0.10)),
        "電價 中位數": float(price.quantile(0.50)),
        "相關係數 充電vs電價": float(charge_mw.corr(price)),
    }


control = pd.DataFrame({k: surplus_profile(v) for k, v in CONTROL.items()})
base_col = control.columns[0]
for col in ["PHS 淨吸收_GWh", "VRE 棄電_GWh"]:
    control.loc[f"{col} 相對對照組倍數"] = np.nan
    control.loc[f"{col} 相對對照組倍數"] = control.loc[col] / control.loc[col, base_col]

control.round(3).to_csv(K.FIG_DIR / "02_table_mustrun_control.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_mustrun_control.csv")
control.round(2)

### 7.4 結論與引用時的注意事項

- **電池**：最佳化後只有 **2.5 MW**（基準情境 2.9 MW），對 260 GW 級的系統是捨入誤差
  等級，在容量地圖與圓餅上等同看不見，**不是繪圖漏畫**。

- **抽蓄無法儲能**：`max_hours = 0`（`PHS_max_hours` 設定未生效的既有 bug），
  SOC 全年恆為 0，**全年放電淨值為 0**。帳面 5,308 MW 完全沒有發揮儲能功能。

- **不要說成 spill**：`storage_units_t.spill` 實測為 **0**。實際機制是**同一時段同時充放電**，
  毛充電 398.6 GWh、毛放電 299.0 GWh，兩者精確滿足 `0.866 × 398.6 = 299.0 ÷ 0.866`，
  淨吸收的 **99.6 GWh 是 75% 來回效率下的損失**，不是被 spill 掉。

- **價格是「歸零」不是「負值」**：最低約 0.01 EUR/MWh，沒有負值——模型裡沒有負成本機組，
  價格下限就是 0。經濟意義（電沒有價值）相同，但用詞要準。

- **這是必發過剩的獨立證據，但 43 倍不能全歸給必發**。kwak_full 有
  **99.6% 的抽蓄充電量**發生在邊際價格 < 1 EUR/MWh 的時段，充電與電價相關係數 **−0.81**，
  訊號本身很明確。不過 7.3 的對照實驗顯示效果要拆成兩段（需求與 VRE 固定時）：

  | 比較 | PHS 淨吸收 | VRE 棄電 | 變動的變因 |
  |---|---|---|---|
  | ① 對照組 → ② uc_lyden | 2.3 → 10.3 GWh（**4.5×**） | 522 → 1,371 GWh（**2.6×**） | **只有必發**（煤 .357 / 核 .45）|
  | ① 對照組 → ③ ramp | 2.3 → 88.9 GWh（**39×**） | 522 → 6,326 GWh（**12×**） | **只有必發**（煤 .35 / 核 **.70**）|
  | ② uc_lyden → ④ kwak_full | 10.3 → 99.6 GWh（**9.7×**） | 1,371 → 9,792 GWh（**7.1×**） | VRE +29%、需求 +6% |

  結論有三層：
  1. **必發限制單獨就足以造成過剩**——在需求與 VRE 完全不變下，光加必發就讓
     抽蓄吸收增加 4.5 倍、棄電增加 2.6 倍。這一點成立，且**獨立於電池的論證**。
  2. **核能的 `p_min_pu` 是主要驅動**——燃煤幾乎沒動（.357 對 .35），
     核能從 0.45 提到 0.70 就讓抽蓄吸收從 4.5 倍跳到 39 倍，效果高度非線性。
  3. **kwak_full 的 43 倍是必發與高 VRE 疊加的結果**，其中 VRE 裝置量增加
     反而是較大的單一貢獻（9.7 倍 vs 4.5 倍）。
     **簡報若要引用「43 倍」，必須同時說明 kwak_full 的 VRE 容量與需求也比基準高**，
     否則會被質疑是不受控的比較。要單獨證明必發效果，請引用 ①→② 這一列。

- **引用時必須排除數值殘留**：config 用 `solver: ipm` + `run_crossover: "off"`，
  內點解不落在頂點，會在**每一個**時段留下微量非零值。因此
  「有充電的時段數 = 100%」這個數字沒有意義，**必須改用電量加權的統計**
  （基準情境有 13% 的充電量落在電價 ≥ 1 的時段，那部分是殘留；kwak_full 只有 0.4%）。

## 8. 風光潛力密度圖（voronoi）

原版只有 onwind / solar，這裡另加 **offwind-ac / offwind-dc**（用離岸 voronoi 區域）。
密度 = 該區域可裝設上限 `p_nom_max` ÷ 區域面積，面積用 **EPSG:5179** 計算
（等面積需求，不能用 PlateCarree 的度數面積）。

In [ ]:
def plot_potential(carrier, kind, cmap, title, fname):
    regions = K.load_regions(10, kind).copy()
    area_km2 = regions.to_crs(K.KR_EQUAL_AREA_CRS).geometry.area * 1e-6

    g = n.generators[n.generators.carrier == carrier]
    p_max = g.groupby("bus").p_nom_max.sum().reindex(regions.index)
    regions["density"] = (p_max / area_km2).fillna(0)

    fig, ax = plt.subplots(figsize=(7.5, 8.5), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_extent(K.KR_EXTENT, crs=ccrs.PlateCarree())
    regions.plot(
        ax=ax, column="density", cmap=cmap, linewidth=0.4, edgecolor="k",
        vmin=0, vmax=float(regions["density"].max()), legend=True,
        legend_kwds={"label": "潛力密度 [MW/km²]", "shrink": 0.7},
        transform=ccrs.PlateCarree(),
    )
    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cartopy.feature.BORDERS.with_scale("50m"), linewidth=0.4)
    ax.set_title(title, fontsize=12, pad=10)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4)
    gl.top_labels = False
    gl.right_labels = False

    K.savefig(fname)
    plt.show()
    return float(p_max.sum()), float((p_max / area_km2).max())


specs = [
    ("onwind", "onshore", "Blues", "陸域風電潛力密度", "02_fig7a_potential_onwind"),
    ("solar", "onshore", "OrRd", "太陽光電潛力密度", "02_fig7b_potential_solar"),
    ("offwind-ac", "offshore", "GnBu", "離岸風電（AC）潛力密度", "02_fig7c_potential_offwind_ac"),
    ("offwind-dc", "offshore", "PuBu", "離岸風電（DC）潛力密度", "02_fig7d_potential_offwind_dc"),
]

pot_rows = []
for carrier, kind, cmap, title, fname in specs:
    total, peak = plot_potential(carrier, kind, cmap, f"KR2036：{title}", fname)
    pot_rows.append({
        "載體": K.carrier_label(carrier),
        "區域": "陸域" if kind == "onshore" else "離岸",
        "潛力上限_GW": total / 1e3,
        "最高密度_MW/km2": peak,
    })

### 8.1 潛力上限彙整

In [ ]:
potential = pd.DataFrame(pot_rows).set_index("載體")
optimal_by_carrier = n.generators.groupby("carrier").p_nom_opt.sum().div(1e3)
potential["最佳化採用_GW"] = [
    optimal_by_carrier.get(c, np.nan) for c in ["onwind", "solar", "offwind-ac", "offwind-dc"]
]
potential["採用率%"] = potential["最佳化採用_GW"] / potential["潛力上限_GW"] * 100

potential.round(2).to_csv(K.FIG_DIR / "02_table_re_potential.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_re_potential.csv")
potential.round(2)

### 8.2 潛力圖判讀

採用率反映的是「模型把該區域的可用面積用掉多少」。要注意這些 `p_nom_max` 是
`build_renewable_profiles` 依土地／海域可用性算出的技術潛力上限，
**已受 `config.yaml` 的 `agg_p_nom_limits`（第 10 次電力供需基本計畫 2036 目標）約束**，
因此採用率高不代表「地不夠用」，而是政策目標容量先綁住了裝置量。

## 9. 容量因數：模型 vs 論文

先前「風電 CF 0.155 對論文 0.263」是從 TWh ÷ GW 反推的，這一節直接算出來。

- **本模型 CF** 由網路直接計算：`實際發電量 ÷ (p_nom_opt × 8760)`，
  另外算一個**可發 CF**（`p_max_pu × p_nom_opt`，即棄電前），
  用來判斷 CF 偏低是「資源不足」還是「被棄電拉低」。
- **論文 CF** 由 repo 既有檔案推導，不從記憶填值：
  `table3_generation_opex.csv` 的 `paper_generation_TWh` ÷
  (`table2_capacity_capex.csv` 的 `paper_capacity_MW` × 8760)。

In [ ]:
PAPER_DIR = K.ROOT / "results/compare/kwak_full_vs_paper"
t2 = pd.read_csv(PAPER_DIR / "table2_capacity_capex.csv", index_col=0, encoding="utf-8-sig")
t3 = pd.read_csv(PAPER_DIR / "table3_generation_opex.csv", index_col=0, encoding="utf-8-sig")

HOURS = 8760
w = n.snapshot_weightings.generators
g = n.generators


def available_pu(idx):
    """可用出力係數：有時序用時序，沒有的用靜態欄位（核能的 0.6 可用率在這裡）。"""
    ts = n.generators_t.p_max_pu.reindex(columns=idx)
    return ts.fillna(g.loc[idx, "p_max_pu"])


rows = []
for carrier in ["nuclear", "coal", "CCGT", "biomass", "onwind", "offwind-ac", "offwind-dc", "solar", "ror"]:
    idx = g.index[g.carrier == carrier]
    if not len(idx):
        continue
    cap = float(g.loc[idx, "p_nom_opt"].sum())
    if cap <= 0:
        continue
    gen_mwh = float(n.generators_t.p[idx].mul(w, axis=0).sum().sum())
    avail_mwh = float(
        (available_pu(idx) * g.loc[idx, "p_nom_opt"]).mul(w, axis=0).sum().sum()
    )
    rows.append({
        "carrier": carrier,
        "容量_GW": cap / 1e3,
        "發電_TWh": gen_mwh / 1e6,
        "本模型 CF": gen_mwh / (cap * HOURS),
        "可用上限 CF": avail_mwh / (cap * HOURS),
    })
cf = pd.DataFrame(rows).set_index("carrier")

# 論文 CF：offwind 在論文表裡是合併的，模型端也合併後才可比
paper_cf = {}
for key in cf.index:
    pk = "offwind" if key.startswith("offwind") else key
    if pk in t2.index and pk in t3.index:
        pcap = float(t2.at[pk, "paper_capacity_MW"])
        pgen = float(t3.at[pk, "paper_generation_TWh"])
        if pcap > 0:
            paper_cf[key] = pgen * 1e6 / (pcap * HOURS)
cf["論文 CF"] = pd.Series(paper_cf)

# 離岸風合併後的可比數字
off_idx = g.index[g.carrier.isin(["offwind-ac", "offwind-dc"])]
off_cap = float(g.loc[off_idx, "p_nom_opt"].sum())
off_gen = float(n.generators_t.p[off_idx].mul(w, axis=0).sum().sum())
cf.loc["offwind（合併）"] = {
    "容量_GW": off_cap / 1e3,
    "發電_TWh": off_gen / 1e6,
    "本模型 CF": off_gen / (off_cap * HOURS),
    "可用上限 CF": np.nan,
    "論文 CF": paper_cf.get("offwind-dc", np.nan),
}

cf.round(4).to_csv(K.FIG_DIR / "02_table_capacity_factor.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_capacity_factor.csv")
cf.round(3)

In [ ]:
plot_cf = cf.drop(index="offwind（合併）").dropna(subset=["論文 CF"])
x = np.arange(len(plot_cf))
width = 0.28

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(x - width, plot_cf["本模型 CF"], width,
       color=[K.carrier_color(i) for i in plot_cf.index], edgecolor="white", label="本模型（實際）")
# 可用上限為 1.0 的機組（未設可用率上限的熱機組）畫出來只是恆真值，會蓋掉重點，故略過
_informative = plot_cf["可用上限 CF"] < 0.999
ax.bar(x[_informative.values], plot_cf.loc[_informative, "可用上限 CF"], width,
       color=[K.carrier_color(i) for i in plot_cf.index[_informative]], alpha=0.45,
       edgecolor="white", hatch="///", label="本模型（可用上限，僅列有設限者）")
ax.bar(x + width, plot_cf["論文 CF"], width,
       color="none", edgecolor="#333333", linewidth=1.4, label="論文（由發電量反推）")

ax.set_xticks(x)
ax.set_xticklabels([K.carrier_label(i) for i in plot_cf.index], rotation=20)
ax.set_ylabel("容量因數")
ax.set_title(f"KR2036 {info['label']}：容量因數對照（模型實際／可用上限／論文）", fontsize=13, pad=10)
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.set_ylim(0, max(1.0, float(plot_cf[["本模型 CF", "可用上限 CF", "論文 CF"]].max().max()) * 1.25))

for xi, (idx_, row) in zip(x, plot_cf.iterrows()):
    ax.text(xi - width, row["本模型 CF"] + 0.015, f"{row['本模型 CF']:.3f}", ha="center", fontsize=7.5)
    ax.text(xi + width, row["論文 CF"] + 0.015, f"{row['論文 CF']:.3f}", ha="center", fontsize=7.5)

K.savefig("02_fig8_capacity_factor")
plt.show()

### 9.1 判讀

關鍵看**陸域風電**：本模型的可用上限 CF與實際 CF 的差距，就是棄電造成的損失；
而可發 CF 與論文 CF 之間若仍有明顯落差，代表差異來自**風資源本身**
（atlite / ERA5 的風速與功率曲線假設），不是調度或棄電造成的。
這一頁是簡報上「唯一沒對上的物理量」的直接支撐——請以圖上的數字為準。

## 10. 淨負載持續曲線

淨負載 = 總負載 − VRE 可發出力（棄電前）。把 8,760 小時由高到低排序，
就能看出兩件事：**尖峰時段需要多少可調度容量**，以及**低谷時段出現多少過剩**
（淨負載 < 0 的區段）。兩個情境並列，可以直接看出必發限制造成的差別。

In [ ]:
def net_load_curve(key):
    m = K.load_network(key)
    w_ = m.snapshot_weightings.generators
    load = m.loads_t.p_set.sum(axis=1)
    g_ = m.generators
    vre = g_.index[g_.carrier.isin(["solar", "onwind", "offwind-ac", "offwind-dc"])]
    avail = (m.generators_t.p_max_pu.reindex(columns=vre).fillna(1.0) * g_.loc[vre, "p_nom_opt"]).sum(axis=1)
    net = load - avail
    hours = float(w_.iloc[0])
    return load, net, hours


fig, ax = plt.subplots(figsize=(11, 6))
stats = []
for key in ["baseline", "kwak_full"]:
    load, net, hours = net_load_curve(key)
    colr = K.SCENARIOS[key]["color"]
    lab = K.SCENARIOS[key]["label"]

    pct = np.linspace(0, 100, len(net))
    ax.plot(pct, np.sort(net.values)[::-1] / 1e3, color=colr, linewidth=2.0, label=f"{lab}：淨負載")
    ax.plot(pct, np.sort(load.values)[::-1] / 1e3, color=colr, linewidth=1.0,
            linestyle="--", alpha=0.55, label=f"{lab}：總負載")

    neg = net < 0
    stats.append({
        "情境": lab,
        "尖峰負載_GW": load.max() / 1e3,
        "淨負載尖峰_GW": net.max() / 1e3,
        "淨負載最低_GW": net.min() / 1e3,
        "淨負載<0 時數": float(neg.sum() * hours),
        "淨負載<0 佔比%": 100 * float(neg.sum()) / len(net),
        "過剩電量_TWh": float((-net[neg]).sum() * hours / 1e6),
    })

ax.axhline(0, color="#444", linewidth=1.0)
ax.set_xlabel("超越時間比例 [%]")
ax.set_ylabel("功率 [GW]")
ax.set_title("淨負載持續曲線（淨負載 = 總負載 − VRE 可發出力）", fontsize=13, pad=10)
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.3)

K.savefig("02_fig9_net_load_duration")
plt.show()

nl = pd.DataFrame(stats).set_index("情境")
nl.round(2).to_csv(K.FIG_DIR / "02_table_net_load.csv", encoding="utf-8-sig")
print("已存檔：", K.FIG_DIR / "02_table_net_load.csv")
nl.round(2)

### 10.1 判讀

- **曲線右端落到 0 以下的區段**是 VRE 可發出力超過**全國總負載**的時數，
  這些電必然要靠棄電、儲能或出口消化。kwak_full 有 78 小時、過剩 1.58 TWh，
  基準情境只有 27 小時、0.51 TWh。
- **但這只是棄電的一部分，不要把兩者畫上等號**：kwak_full 全年 VRE 棄電是
  **9.79 TWh**，遠大於淨負載 < 0 的 1.58 TWh。差額來自「全國總量還沒過剩，
  但局部已經塞不下」的情形——熱機組必發下限讓可調度機組降不下來，
  加上節點間輸電容量限制，使得棄電在淨負載仍為正的時段就已經發生。
  換句話說，**這張圖低估了實際的彈性需求**。
- **曲線左端（淨負載尖峰）**決定系統要保留多少可調度容量，
  這個數字不會因為多裝 VRE 而下降，是「彈性需求從哪裡來」的分母
  （kwak_full 99.7 GW，僅比尖峰負載 114.6 GW 低 13%）。
- **兩情境的差異有三個成因，不能單獨歸因於任何一個**：年需求（667.3 對 707.0 TWh）、
  VRE 裝置量（102.9 對 132.8 GW）、熱機組必發限制（無對有）。
  本圖同時反映這三者——這也是為什麼 kwak_full 的總負載曲線（虛線）整條都比基準高。
  要單獨看必發限制的效果，請用第 7.3 節需求與 VRE 都固定的對照實驗。